In [1]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

import sys
import torch
from nnsight import LanguageModel

sys.path.append("./mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

torch.set_grad_enabled(False)

In [2]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir="./jbloom_dictionaries")

    save_path = f"./jbloom_dictionaries/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [3]:
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

Loaded GPT


In [4]:
kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)
# mixtral = LanguageModel("mistralai/Mistral-7B-Instruct-v0.2", device_map="cuda:0", dispatch=True)

print("Loaded Mixtral in 4bit")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [5]:
import importlib 
import agent.Environment

importlib.reload(agent.Environment)

env = agent.Environment.Environment(mixtral, gpt, sae_list)

In [6]:
layer = 10
index = 6536

prompts = [
    " (getting back 87 cents on the dollar in 2010.) In comparison, the average state gets",
    " (Dungeons and Dragons figurines off the kitchen table).ĊĊThe other day I noteds",
    ", (appears to be in much the same boat.) Yes, our political leaders are perfectly",
]

env(
    prompts = prompts,
    layer = layer,
    index = index
)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Loaded features


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/share/u/caden/.conda/envs/interp/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:226: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(f'Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.')
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A

['(found **three dollars** in **2022** at the **bank** ).', '(the **kitchen** table holds **Dungeons & Dragons** figurines).s_three', '(the **captain** is in the **same boat**)']


┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Content                                                                                                  ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ User      │ You are a meticulous AI researcher conducting a high-stakes investigation on neurons in a large language │
│           │ model. Your task is to understand what features of the input text cause a specific neuron to activate.   │
│           │                                                                                                          │
│           │ You will be given a list of text samples containing tokens on which the neuron activates strongly. The   │
│           │ specific tokens which caused the neuron to activate strongly will appear between bars like ** this**. If │
│           │ multiple tokens cause the neuron to activate strongly, the entire sequence will be contained between     │
│           │ bars ** just like this**.                                                                                │
│           │                                                                                                          │
│           │ You will be given multiple samples on which a neuron activates strongly. For each sample in turn, note   │
│           │ down a few features that the text possesses, even if you don't initially think they are important.       │
│           │                                                                                                          │
│           │ Once you have written down a few notes for each text sample, summarize what highly-activating samples    │
│           │ have in common. Finally, use your findings to produce a plausible explaination for what causes the       │
│           │ neuron to fire.                                                                                          │
│           │                                                                                                          │
│           │ Sample 0:<|endoftext|> (getting back 87 cents on the** dollar** in** 2010**.) In comparison, the average │
│           │ state gets                                                                                               │
│           │ Sample 1:<|endoftext|> (Dungeons and Dragons figurines off the** kitchen**** table**).����The other day  │
│           │ I noteds                                                                                                 │
│           │ Sample 2:<|endoftext|>, (appears to be in much the same** boat**.) Yes, our political leaders are        │
│           │ perfectly                                                                                                │
│           │                                                                                                          │
│ Assistant │ Sample 0 features:                                                                                       │
│           │                                                                                                          │
│           │ * The presence of the currency-related term "dollar" and a numerical value "87 cents".                   │
│           │ * The usage of a year "2010".                                                                            │
│           │                                                                                                          │
│           │ Sample 1 features:                                                                                       │
│           │                                                                                                          │
│           │ * The inclusion of an activity related to removing items from a specific location ("off the kitchen      │
│      

'Based on the analysis of the provided text samples, it appears that the neuron fires when the input text contains certain features. These features include references to specific objects, time indicators such as years, locations, and monetary values. Here are some possible explanations for the neuron activation:\n\n1. Object reference: The neuron may be associated with recognizing the presence of specific objects mentioned in the text, such as "dollars," "kitchen table," or "D'